<a href="https://colab.research.google.com/github/Sakinashmadi87/Static-Website/blob/main/Copy_of_Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Bibliotheken installieren

In [ ]:
# Installiere notwendige Bibliotheken
!pip install torch transformers wandb datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Tiny Shakespeare laden

In [ ]:
import requests

# Tiny Shakespeare direkt von der URL laden
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text

# Daten vorbereiten

In [ ]:
from transformers import GPT2Tokenizer
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Tokenizer initialisieren
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Text tokenisieren
tokens = np.array(tokenizer.encode(text))

# Dataset definieren
class TextDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.tokens = tokens
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):
        return (self.tokens[idx:idx+self.seq_len],
                self.tokens[idx+1:idx+self.seq_len+1])

# Dataloader erstellen
seq_len = 64  # Kürzere Sequenz für CPU
batch_size = 8  # Kleine Batch-Größe
dataset = TextDataset(tokens, seq_len=seq_len)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Überprüfe die Größe des Datensatzes
print(f"Anzahl der Samples: {len(dataset)}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


Anzahl der Samples: 337961


# Modell definieren

In [ ]:
import torch
import torch.nn as nn
import math

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Decoder-only Transformer
class DecoderLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward=4*d_model)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        self.fc = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
        self.max_len = max_len

    def forward(self, src, tgt_mask=None):
        # src: (batch_size, seq_len) -> (seq_len, batch_size, d_model)
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        src = src.transpose(0, 1)  # (seq_len, batch_size, d_model)

        # Erstelle Maske
        if tgt_mask is None:
            tgt_mask = self.generate_mask(src.size(0)).to(src.device)

        # TransformerDecoder erwartet (tgt_len, batch_size, d_model)
        output = self.transformer_decoder(src, memory=src, tgt_mask=tgt_mask)
        output = output.transpose(0, 1)  # Zurück zu (batch_size, seq_len, d_model)
        return self.fc(output)

    def generate_mask(self, sz):
        # Erstelle autoregressive Maske
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1)
        return mask
# Modell initialisieren
vocab_size = tokenizer.vocab_size  # ~50,257
model = DecoderLanguageModel(vocab_size=vocab_size, d_model=256, nhead=4, num_layers=4)

# Training mit wandb

In [ ]:
import torch.optim as optim
import wandb

# Wandb initialisieren
wandb.init(project="decoder-language-model-colab", config={
    "epochs": 5,
    "batch_size": 8,
    "d_model": 256,
    "num_layers": 4,
    "nhead": 4,
    "seq_len": 64
})

# Modell und Optimierer
device = torch.device("cpu")
model = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

epochs = 5
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in dataloader:
        src, tgt = batch
        src, tgt = src.to(device), tgt.to(device)

        optimizer.zero_grad()
        output = model(src)  # tgt_mask wird intern erstellt

        loss = criterion(output.view(-1, vocab_size), tgt.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(dataloader)
    perplexity = math.exp(avg_train_loss)
    wandb.log({"epoch": epoch, "train_loss": avg_train_loss, "perplexity": perplexity})
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Perplexity: {perplexity:.4f}")

    # Modell nach jeder Epoche speichern
    torch.save(model.state_dict(), f"decoder_model_cpu_epoch_{epoch+1}.pt")

    # Textgenerierung
def generate_text(model, tokenizer, prompt, max_len=50):
    model.eval()
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)
    generated = input_ids

    for _ in range(max_len):
        tgt_mask = model.generate_mask(generated.size(1)).to(device)
        with torch.no_grad():
            output = model(generated, tgt_mask)
        next_token = output[:, -1, :].argmax(dim=-1).unsqueeze(0)
        generated = torch.cat((generated, next_token), dim=1)

    return tokenizer.decode(generated.squeeze().tolist())

prompt = "To be or not to be"
generated_text = generate_text(model, tokenizer, prompt)
print(f"Generated: {generated_text}")

# Wandb abschließen
wandb.finish()

KeyboardInterrupt: 